# Larger encoders on typed-decisions: mmBERT-base, ModernBERT-large and Ettin-400m

Fully tuned ModernBERT-base reached 0.787 test accuracy (by `evaluate`; about 0.778 as the leaderboard scores it) on
[typed-decisions](https://huggingface.co/datasets/LocalLLaMA/typed-decisions). This notebook
trains three candidates to beat it, with the same data and setup, and scores them the way the
dataset's leaderboard does:

| step | model | parameters (non-embedding) | expected time, Colab L4 | A100 |
|---|---|---|---|---|
| 1 | `jhu-clsp/mmBERT-base` | 307M (~110M) | 20–25 min | 8–10 min |
| 2 | `answerdotai/ModernBERT-large` | 395M (~340M) | 50–75 min | 20–25 min |
| 3 | `jhu-clsp/ettin-encoder-400m` | 400M (~350M) | 50–75 min | 20–25 min |

> **⚠️ Use a GPU runtime** (Runtime → Change runtime type → L4 or A100) and connect Google Drive
> when asked. Each run autosaves to Drive while it trains, and its results are written there when
> it finishes. If the runtime disconnects, run the notebook again from the top: finished steps are
> read back from Drive, and an unfinished one continues from its last save.

In [ ]:
# On Google Colab, or anywhere sysone isn't installed yet, install it from PyPI.
import importlib.util

if importlib.util.find_spec("sysone") is None:
    %pip install -q "sysone>=0.4"
if importlib.util.find_spec("datasets") is None:
    %pip install -q datasets

In [ ]:
import json
import math
import statistics
import time
from pathlib import Path

import torch
from datasets import load_dataset

from sysone import EncoderModel, baselines, compare

# Results and checkpoints go to Google Drive on Colab, so a disconnect loses nothing; elsewhere
# they go to a local folder.
try:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/sysone-larger-encoders")
except ImportError:
    OUT = Path("sysone-larger-encoders")
OUT.mkdir(parents=True, exist_ok=True)
print("saving to", OUT)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be very slow)")

## Load the data

All four workflows: 1,200 training and 400 test records, five questions each, with soft labels
(each answer is the raters' probability spread). `state`, `questions` and `gold` are JSON text,
which sysone decodes itself; the workflow is the row's `category`, so validation is split evenly
across workflows.

In [ ]:
def load_rows(split):
    ds = load_dataset("LocalLLaMA/typed-decisions", "all", split=split)
    return [{"state": r["state"], "questions": r["questions"], "gold": r["gold"], "category": r["workflow"]}
            for r in ds]


train = load_rows("train")
test = load_rows("test")
print("%d training records, %d test records" % (len(train), len(test)))

## Scoring the way the leaderboard does

`model.evaluate` gives sysone's metrics. The leaderboard computes its own, with the formulas from
the dataset's `bench/score_with_jobs.py`, so the function below copies them:

- **accuracy**: the predicted top option is the raters' top option (a tie goes to the first option);
- **kl_from_gold**: KL divergence from the raters' distribution to the prediction;
- **brier**: squared distance to the raters' distribution, summed over options;
- **ece**: calibration error of the top option, in 10 bins.

It also times predictions: all 400 test records in batches, and 50 records one at a time (the
latency a single request sees).

In [ ]:
def leaderboard_scores(model, rows):
    results = model.predict([{"state": r["state"], "questions": r["questions"]} for r in rows])
    accs, kls, briers, confs = [], [], [], []
    for row, result in zip(rows, results):
        for qid, gold in json.loads(row["gold"]).items():
            answer = result["answers"][qid]
            # sysone 0.4 gives yes/no answers their probabilities; earlier versions give only P(yes)
            probs = answer.get("probabilities") or {"false": 1 - answer["noul"], "true": answer["noul"]}
            options = list(gold["probabilities"])
            g = [float(gold["probabilities"][o]) for o in options]
            p = [float(probs.get(o, 0.0)) for o in options]
            total = sum(p)
            p = [v / total for v in p] if total > 0 else [1 / len(p)] * len(p)
            accs.append(p.index(max(p)) == g.index(max(g)))
            kls.append(sum(gi * math.log(gi / max(pi, 1e-9)) for gi, pi in zip(g, p) if gi > 0))
            briers.append(sum((pi - gi) ** 2 for pi, gi in zip(p, g)))
            confs.append(max(p))
    n = len(accs)
    bins = [[0, 0, 0.0] for _ in range(10)]  # count, hits, confidence sum
    for ok, conf in zip(accs, confs):
        b = bins[min(int(conf * 10), 9)]
        b[0] += 1
        b[1] += ok
        b[2] += conf
    ece = sum(t / n * abs(h / t - c / t) for t, h, c in bins if t)
    return {"accuracy": sum(accs) / n, "kl_from_gold": sum(kls) / n, "brier": sum(briers) / n, "ece": ece}


def prediction_times(model, rows):
    start = time.perf_counter()
    model.predict([{"state": r["state"], "questions": r["questions"]} for r in rows])
    batched = (time.perf_counter() - start) / len(rows) * 1000
    single = []
    for r in rows[:50]:
        start = time.perf_counter()
        model.predict(r["state"], json.loads(r["questions"]))
        single.append((time.perf_counter() - start) * 1000)
    return {"ms_per_record_batched": batched, "ms_per_record_single_p50": statistics.median(single)}

## Train, score and save, resumably

`run` trains one model and writes everything to `OUT/<name>/`:

- `checkpoint/`: autosaves while training (`checkpoint_every` optimizer steps), with the state
  to resume from;
- `model/`: the finished model without its training state, ready for `push_to_hub` or sharing;
- `results.json`: sysone's report, the leaderboard scores, the times and peak GPU memory.

Run again after a disconnect and it picks up where it was: a finished model's results are read
back, and an unfinished run continues from its last checkpoint. Training minutes count only the
session that finished the run.

In [ ]:
def run(name, model_name, **params):
    folder = OUT / name
    results_file = folder / "results.json"
    if results_file.exists():
        print("%s: already done, reading %s" % (name, results_file))
        return json.loads(results_file.read_text())

    checkpoint = folder / "checkpoint"
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    resumed = (checkpoint / "run.json").exists()
    if resumed:
        info = json.loads((checkpoint / "run.json").read_text())
        print("%s: resuming from step %d (%s)" % (name, info["step"], info["reason"]))
        model = EncoderModel.load(checkpoint)
        if info["reason"] != "finished":
            model.fit(train)
    else:
        model = EncoderModel(model_name, checkpoint_dir=str(checkpoint), **params)
        model.fit(train)
    fit_minutes = (time.perf_counter() - start) / 60

    report = model.evaluate(test)
    results = {
        "name": name,
        "model_name": model_name,
        "params": {k: v for k, v in params.items()},
        "fit_minutes": fit_minutes,
        "resumed": resumed,
        "peak_gpu_gb": torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else None,
        "leaderboard": leaderboard_scores(model, test),
        **prediction_times(model, test),
        "report": report,
        "calibration": model.report_.get("calibration") if model.report_ else None,
    }
    model.save(folder / "model", training_state=False)
    results_file.write_text(json.dumps(results, indent=1, default=str))
    print("%s: leaderboard accuracy %.4f, %.1f minutes" % (name, results["leaderboard"]["accuracy"], fit_minutes))
    return results


# Settings shared by both runs. Validation runs about ten times per epoch; training stops once
# validation loss hasn't improved for 3 epochs and keeps the best weights, so `epochs` is an
# upper bound.
COMMON = dict(tuning="full", epochs=8, batch_size=32, seed=0, checkpoint_every=300)

runs = {}

## Before training: baselines

- **random guess**: every option equally likely;
- **average answer**: for each workflow question, the raters' average distribution in the
  training data. Every record of a workflow has the same questions, so this is a strong bar.

In [ ]:
results = baselines(test, train=train)
compare(results, by="overall")

## Step 1: mmBERT-base

A multilingual ModernBERT (1,800+ languages) with ModernBERT-base's body: 22 layers, 768 wide.
Most of its 307M parameters are its 256k-token vocabulary, which stays frozen
(`train_embeddings=False`), so it trains about as fast as ModernBERT-base. The learning rate is
ModernBERT-base's measured 5e-5, since the body is the same size.

In [ ]:
runs["mmBERT-base"] = run("mmbert-base", "jhu-clsp/mmBERT-base", backbone_lr=5e-5, **COMMON)

## Step 2: ModernBERT-large

28 layers, 1,024 wide: about three times ModernBERT-base's compute. Larger models usually want a
lower learning rate, so this uses 3e-5 (not yet measured; if validation loss stops improving in
the first two epochs, try 5e-5).

In [ ]:
runs["ModernBERT-large"] = run("modernbert-large", "answerdotai/ModernBERT-large", backbone_lr=3e-5, **COMMON)

## Step 3: Ettin-encoder-400m

The same shape as ModernBERT-large (28 layers, 1,024 wide, the same tokenizer), from newer
pretraining. sysone runs it through the ModernBERT architecture. Up to 10 epochs, since it can
keep improving past epoch 6; early stopping ends it sooner if not. In one run on an L4 it reached
0.8025, in 59 minutes.

In [ ]:
runs["Ettin-400m"] = run("ettin-400m", "jhu-clsp/ettin-encoder-400m", backbone_lr=3e-5, **{**COMMON, "epochs": 10})

## Summary

The leaderboard's metrics and the costs, one row per model, with the earlier ModernBERT-base run
for reference (its Brier, ECE and KL were not recorded).

In [ ]:
import pandas as pd

rows = {"ModernBERT-base (earlier run, seed 0)": {"accuracy": 0.787, "fit_minutes": 21.0}}
for label, r in runs.items():
    rows[label] = {**r["leaderboard"], "fit_minutes": r["fit_minutes"], "peak_gpu_gb": r["peak_gpu_gb"],
                   "ms_per_record_batched": r["ms_per_record_batched"],
                   "ms_per_record_single_p50": r["ms_per_record_single_p50"]}
summary = pd.DataFrame.from_dict(rows, orient="index")
summary.round(4)

sysone's own metrics, against the baselines (sysone's `ece` uses 15 bins, and its accuracy splits
credit when the predicted options tie, so they differ slightly from the leaderboard's):

In [ ]:
for label, r in runs.items():
    results[label] = r["report"]
compare(results, by="overall")

In [ ]:
compare(results, by="type")      # accuracy per question type

In [ ]:
compare(results, by="category")  # accuracy per workflow

## Reading the results

The leaderboard's table of models fine-tuned on `train` (on the
[dataset card](https://huggingface.co/datasets/LocalLLaMA/typed-decisions)), for comparison:

| model | size | accuracy |
|---|---|---|
| Decider-4B | 4B | 0.8205 |
| OpenDecider-nano | 400M encoder | 0.796 |
| Phocinae-Largha (mmBERT-small) | 144M | 0.791 |
| OpenDecider-medium | 30B MoE | 0.788 |
| soft-decider-421m | 421M | 0.774 |
| the maintainers' ModernBERT-base | 149M | 0.646 |

- One seed is one draw: before submitting, train the best model with two more seeds
  (`seed=1`, `seed=2`) and report the mean.
- Accuracy is measured against the raters' top answer, and raters often disagree, so the ceiling
  is well below 100%; KL and Brier show how close the probabilities are to the raters' spread.
- Each finished model is in `OUT/<name>/model`; `EncoderModel.load(...)` reloads it, and on
  sysone 0.4 `model.push_to_hub("you/name")` shares it.